In [ ]:
!pip -q install numpy
!pip -q uninstall -y cupy-cuda12x || true
!pip -q install cupy-cuda13x || true
!rm -rf /content/DanyAI
!git clone -q https://github.com/DanyaVoredom/DanyAI.git /content/DanyAI
%cd /content/DanyAI
!pip -q install -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv || true
print("setup done")

In [ ]:
import base64, os, pathlib, urllib.request
data_raw = "https://gist.githubusercontent.com/DanyaVoredom/6fc3e3be94e69991f1bcbf45eab0e3a7/raw/danyai_data.bundle"
text = urllib.request.urlopen(data_raw).read().decode()
os.makedirs("data", exist_ok=True)
cur = None
buf = []
def flush():
    global cur, buf
    if cur is None:
        return
    b64 = "".join(buf)
    pathlib.Path("data", cur).write_bytes(base64.b64decode(b64))
    print("wrote", cur, len(b64) * 3 // 4)
    cur = None
    buf = []
for line in text.splitlines():
    if line.startswith("@@FILE "):
        flush()
        cur = line.split()[1]
    elif line.startswith("@@END"):
        flush()
    else:
        buf.append(line.strip())
flush()
print("data", sorted(os.listdir("data")))

In [ ]:
import json, os
os.environ["DANYAI_GPU"] = "1"
import numpy as np
from danyai.backend import GPU
print("GPU", GPU)
cfg = json.load(open("configs/modern.json"))
cfg.update(block_size=256, batch_size=32, steps=2000, lr=0.002, warmup=200)
cfg["name"] = "DanyAI-Modern-Colab"
cfg["log_every"] = 1
json.dump(cfg, open("configs/colab.json", "w"), indent=2)
print(cfg, flush=True)
!DANYAI_GPU=1 python -m danyai.train --config configs/colab.json --data data --out runs/colab
!DANYAI_GPU=1 python -m danyai.sample --ckpt runs/colab/best.npz --tok data/tokenizer.json --prompt "Once upon a time" --tokens 120 --temp 0.8
print("train done")

In [ ]:
!DANYAI_GPU=1 python -m danyai.sft --ckpt runs/colab/best.npz --data data/sft.jsonl --tok data/tokenizer.json --out runs/sft --steps 600 --batch 4 --lr 2e-4
!DANYAI_GPU=1 python -m danyai.sample --ckpt runs/sft/best.npz --tok data/tokenizer.json --prompt "<|user|>Write a short story about a cat.<|assistant|>" --tokens 120 --temp 0.7
print("sft done")

In [ ]:
import base64, pathlib
for name in ["runs/colab/best.npz", "runs/sft/best.npz", "data/tokenizer.json", "data/meta.json", "configs/colab.json"]:
    p = pathlib.Path(name)
    if not p.exists():
        print("@@MISSING", name)
        continue
    b = base64.b64encode(p.read_bytes()).decode()
    print(f"@@BEGIN {name} {len(b)}")
    for i in range(0, len(b), 50000):
        print(b[i:i + 50000])
    print(f"@@END {name}")
print("pack done")